# Short-horizon rebuild notebook (1h, 6h, 12h, 24h)

This notebook rebuilds the eight-model comparison from the original project using the same overall methods, but with shorter forecast horizons.

The model set is:

1. Elastic Net  
2. Random Forest  
3. LSTM  
4. CNN–GRU–LSTM  
5. GraphWaveNet  
6. GraphWaveNet–GRU  
7. GraphWaveNet–LSTM  
8. GraphWaveNet–GRU–LSTM  

The methodological goal is to keep the project aligned with the original notebooks:

- same split dates
- same station set
- same feature channels
- same train-only scaling
- same strict sliding-window logic
- same evaluation style in original traffic-flow units

The only forecasting change is:

- old setup: `OUT_LEN = 72`, report `12, 24, 48, 72`
- new setup: `OUT_LEN = 24`, report `1, 6, 12, 24`


In [1]:
from pathlib import Path
import copy
import gc
import json
import random
import time

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import MultiTaskElasticNet
from sklearn.ensemble import RandomForestRegressor
from joblib import Parallel, delayed

from IPython.display import display


def set_seed(seed: int = 42, deterministic: bool = True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


SEED = 42
set_seed(SEED, deterministic=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------------
# Raw files
# -------------------------
TRAFFIC_CSV = Path("cleaned_traffic_data.csv")
META_XLSX   = Path("pems_output.xlsx")

# -------------------------
# Existing strict artifact from the original project
# -------------------------
BASE_STRICT_DATASET = Path("artifacts/pems_graph_dataset_strict.npz")

# -------------------------
# Split boundaries (same as original project)
# -------------------------
TRAIN_END = pd.Timestamp("2024-11-15 23:59:59")
VAL_END   = pd.Timestamp("2024-11-30 23:59:59")

# -------------------------
# Short-horizon setup
# -------------------------
IN_LEN = 24
OUT_LEN = 24
EVAL_HORIZONS = [1, 6, 12, 24]

assert max(EVAL_HORIZONS) <= OUT_LEN, "The largest evaluation horizon must be <= OUT_LEN."

SHORT_TAG = "short_h1_6_12_24"

OUT_DIR = Path("artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SHORT_DATASET = OUT_DIR / f"pems_graph_dataset_strict_{SHORT_TAG}.npz"
RESULTS_SUMMARY_SHORT = OUT_DIR / f"results_summary_{SHORT_TAG}.csv"

print("BASE_STRICT_DATASET:", BASE_STRICT_DATASET)
print("SHORT_DATASET:", SHORT_DATASET)
print("RESULTS_SUMMARY_SHORT:", RESULTS_SUMMARY_SHORT)

Torch: 2.1.1+cu121
Device: cuda
GPU: Quadro P5000
BASE_STRICT_DATASET: artifacts/pems_graph_dataset_strict.npz
SHORT_DATASET: artifacts/pems_graph_dataset_strict_short_h1_6_12_24.npz
RESULTS_SUMMARY_SHORT: artifacts/results_summary_short_h1_6_12_24.csv


## Rebuild the strict dataset for 24-step forecasting

This is the key short-horizon conversion step.

To stay methodologically faithful to the original project, we do not rebuild the whole cleaned dataset from scratch here.  
Instead, we:

- load the already-built strict artifact from the original notebook
- keep the same `X`, `Y`, adjacency, stations, timestamps, and train-only statistics
- recompute only the valid start indices and split assignments for `OUT_LEN = 24`

That way, the study changes only in the forecast length and evaluation horizons.

In [2]:
assert BASE_STRICT_DATASET.exists(), (
    f"Missing {BASE_STRICT_DATASET}. "
    "Run the original notebook once so the strict base artifact exists."
)

base = np.load(BASE_STRICT_DATASET, allow_pickle=True)

X_base = base["X"].astype(np.float32)
Y_base = base["Y"].astype(np.float32)
A_base = base["A"].astype(np.float32)
stations_base = base["stations"]
timestamps_base = pd.to_datetime(base["timestamps"])

flow_mean_base = base["flow_mean"].astype(np.float32)
flow_std_base  = base["flow_std"].astype(np.float32)
speed_mean_base = base["speed_mean"].astype(np.float32)
speed_std_base  = base["speed_std"].astype(np.float32)

base_in_len = int(np.array(base["in_len"]).item())
print("Base strict artifact loaded.")
print("Base X:", X_base.shape, "Base Y:", Y_base.shape)
print("Base IN_LEN:", base_in_len)

assert base_in_len == IN_LEN, (
    f"Expected base IN_LEN={IN_LEN}, found {base_in_len}. "
    "This notebook assumes the original project used 24 hours of input."
)

T_total = X_base.shape[0]
max_start = T_total - (IN_LEN + OUT_LEN) + 1
starts = np.arange(max_start, dtype=np.int32)

out_start_times = timestamps_base[starts + IN_LEN]
out_end_times   = timestamps_base[starts + IN_LEN + OUT_LEN - 1]

train_starts_short = starts[out_end_times <= TRAIN_END]
val_starts_short   = starts[(out_start_times > TRAIN_END) & (out_end_times <= VAL_END)]
test_starts_short  = starts[out_start_times > VAL_END]

print("Short-horizon strict window counts:")
print("train:", len(train_starts_short))
print("val:  ", len(val_starts_short))
print("test: ", len(test_starts_short))

np.savez_compressed(
    SHORT_DATASET,
    X=X_base.astype(np.float32),
    Y=Y_base.astype(np.float32),
    A=A_base.astype(np.float32),
    stations=stations_base,
    timestamps=np.array(timestamps_base.astype("datetime64[ns]")),
    train_starts=train_starts_short,
    val_starts=val_starts_short,
    test_starts=test_starts_short,
    in_len=np.array([IN_LEN], dtype=np.int32),
    out_len=np.array([OUT_LEN], dtype=np.int32),
    flow_mean=flow_mean_base,
    flow_std=flow_std_base,
    speed_mean=speed_mean_base,
    speed_std=speed_std_base,
)

print("Saved short-horizon strict dataset to:", SHORT_DATASET)

Base strict artifact loaded.
Base X: (2208, 1821, 6) Base Y: (2208, 1821)
Base IN_LEN: 24
Short-horizon strict window counts:
train: 1057
val:   337
test:  721
Saved short-horizon strict dataset to: artifacts/pems_graph_dataset_strict_short_h1_6_12_24.npz


## Load the short-horizon shared tensors

We keep the same shared representation used in the original notebooks:

- `X` with shape `(T, N, F)`
- `Y` with shape `(T, N)`

The feature channels remain:

1. flow  
2. speed  
3. hour_sin  
4. hour_cos  
5. dow_sin  
6. dow_cos  

Only the output sequence length changes from `72` to `24`.

In [3]:
assert SHORT_DATASET.exists(), f"Missing {SHORT_DATASET}. Run the rebuild cell above first."

data = np.load(SHORT_DATASET, allow_pickle=True)

X = data["X"].astype(np.float32)          # (T, N, F)
Y = data["Y"].astype(np.float32)          # (T, N)
A = data["A"].astype(np.float32)          # (N, N)

stations = data["stations"]
timestamps = pd.to_datetime(data["timestamps"])

train_starts = data["train_starts"].astype(np.int64)
val_starts   = data["val_starts"].astype(np.int64)
test_starts  = data["test_starts"].astype(np.int64)

IN_LEN  = int(np.array(data["in_len"]).item())
OUT_LEN = int(np.array(data["out_len"]).item())

flow_mean  = data["flow_mean"].astype(np.float32)
flow_std   = data["flow_std"].astype(np.float32)
speed_mean = data["speed_mean"].astype(np.float32)
speed_std  = data["speed_std"].astype(np.float32)

flow_std  = np.maximum(flow_std, 1e-6).astype(np.float32)
speed_std = np.maximum(speed_std, 1e-6).astype(np.float32)

T, N, Fdim = X.shape
print("X:", X.shape, "(T,N,F)")
print("Y:", Y.shape, "(T,N)")
print("A:", A.shape)
print("IN_LEN / OUT_LEN:", IN_LEN, OUT_LEN)
print("Stations:", N)
print("train / val / test starts:", len(train_starts), len(val_starts), len(test_starts))


def time_encoding(dt_index: pd.DatetimeIndex) -> np.ndarray:
    hours = dt_index.hour.values
    dow   = dt_index.dayofweek.values
    hour_sin = np.sin(2 * np.pi * hours / 24.0)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)
    dow_sin  = np.sin(2 * np.pi * dow / 7.0)
    dow_cos  = np.cos(2 * np.pi * dow / 7.0)
    return np.stack([hour_sin, hour_cos, dow_sin, dow_cos], axis=1).astype(np.float32)


TF_all = time_encoding(pd.DatetimeIndex(timestamps))  # (T, 4)

# Scale only the continuous sensor channels using train-only stats saved in the artifact.
X_scaled = X.copy()
X_scaled[:, :, 0] = (X_scaled[:, :, 0] - flow_mean[None, :]) / flow_std[None, :]
X_scaled[:, :, 1] = (X_scaled[:, :, 1] - speed_mean[None, :]) / speed_std[None, :]

Y_scaled = (Y - flow_mean[None, :]) / flow_std[None, :]

# Conv-style layout reused in the original notebooks: (F, N, T)
X_fnt = np.transpose(X_scaled, (2, 1, 0)).copy()

print("TF_all:", TF_all.shape)
print("X_fnt:", X_fnt.shape)
print("Scaled target mean/std:", float(Y_scaled.mean()), float(Y_scaled.std()))

X: (2208, 1821, 6) (T,N,F)
Y: (2208, 1821) (T,N)
A: (1821, 1821)
IN_LEN / OUT_LEN: 24 24
Stations: 1821
train / val / test starts: 1057 337 721
TF_all: (2208, 4)
X_fnt: (6, 1821, 2208)
Scaled target mean/std: -781.403564453125 30704.76953125


## Shared dataset and dataloaders for all deep models

All deep models use the same sliding-window dataset:

- `x`: `(F, N, IN_LEN)`
- `y`: `(OUT_LEN, N)` in scaled flow units
- `tf`: `(OUT_LEN, 4)` future calendar features for the target steps

This matches the later clean sections of the original project.

In [4]:
class FastPemsWindowDataset(Dataset):
    def __init__(self, X_fnt, Y_scaled, TF_all, starts, in_len, out_len):
        self.X_fnt = X_fnt
        self.Y = Y_scaled
        self.TF = TF_all
        self.starts = np.asarray(starts, dtype=np.int64)
        self.in_len = int(in_len)
        self.out_len = int(out_len)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, idx):
        s = int(self.starts[idx])
        x = self.X_fnt[:, :, s:s + self.in_len]                    # (F, N, IN)
        y = self.Y[s + self.in_len:s + self.in_len + self.out_len, :]  # (OUT, N)
        tf = self.TF[s + self.in_len:s + self.in_len + self.out_len, :] # (OUT, 4)

        return (
            torch.from_numpy(x).float(),
            torch.from_numpy(y).float(),
            torch.from_numpy(tf).float(),
        )


train_ds = FastPemsWindowDataset(X_fnt, Y_scaled, TF_all, train_starts, IN_LEN, OUT_LEN)
val_ds   = FastPemsWindowDataset(X_fnt, Y_scaled, TF_all, val_starts,   IN_LEN, OUT_LEN)
test_ds  = FastPemsWindowDataset(X_fnt, Y_scaled, TF_all, test_starts,  IN_LEN, OUT_LEN)

BATCH_SIZE = 8

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

xb, yb, tfb = next(iter(train_loader))
print("Batch x:", xb.shape)
print("Batch y:", yb.shape)
print("Batch tf:", tfb.shape)

Batch x: torch.Size([8, 6, 1821, 24])
Batch y: torch.Size([8, 24, 1821])
Batch tf: torch.Size([8, 24, 4])


## Shared evaluator and run-output utilities

All deep models are trained on scaled targets and evaluated in original traffic-flow units.

The evaluation rule is unchanged from the original notebooks:

- horizon `h` maps to output index `h - 1`
- MAE and RMSE are computed in original vehicles/hour units
- early stopping monitors average validation MAE across the selected horizons

This cell also writes a fresh short-horizon summary CSV so the new results do not overwrite the old 72-hour study.

In [5]:
flow_mean_t = torch.tensor(flow_mean, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
flow_std_t  = torch.tensor(flow_std,  dtype=torch.float32, device=DEVICE).view(1, 1, -1)


def print_metrics(title: str, metrics: dict):
    print("\n" + title)
    for h in sorted(metrics.keys()):
        print(f"  {h:>3}h  MAE={metrics[h]['MAE']:.3f}  RMSE={metrics[h]['RMSE']:.3f}")


def avg_mae(metrics: dict) -> float:
    return float(np.mean([metrics[h]["MAE"] for h in metrics]))


@torch.inference_mode()
def eval_horizons_fast(model, loader, eval_horizons=EVAL_HORIZONS):
    model.eval()
    acc = {h: {"abs": 0.0, "sq": 0.0, "count": 0} for h in eval_horizons}

    for xb, yb, tfb in tqdm(loader, desc="Eval", leave=False):
        xb  = xb.to(DEVICE, non_blocking=True)
        yb  = yb.to(DEVICE, non_blocking=True)
        tfb = tfb.to(DEVICE, non_blocking=True)

        pred = model(xb, tfb)  # scaled (B, OUT, N)

        pred_u = pred * flow_std_t + flow_mean_t
        true_u = yb   * flow_std_t + flow_mean_t

        for h in eval_horizons:
            idx = h - 1
            err = pred_u[:, idx, :] - true_u[:, idx, :]
            acc[h]["abs"] += float(err.abs().sum())
            acc[h]["sq"]  += float((err ** 2).sum())
            acc[h]["count"] += err.numel()

    metrics = {}
    for h in eval_horizons:
        mae = acc[h]["abs"] / acc[h]["count"]
        rmse = (acc[h]["sq"] / acc[h]["count"]) ** 0.5
        metrics[h] = {"MAE": float(mae), "RMSE": float(rmse)}
    return metrics


def make_run_dir(model_name: str, tag: str = SHORT_TAG) -> Path:
    stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    run_dir = Path("artifacts/runs") / f"{stamp}_{model_name}_{tag}"
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


def save_metrics_files(run_dir: Path, split_name: str, metrics: dict):
    save_json(run_dir / f"{split_name}_metrics.json", metrics)
    rows = [{"horizon_h": h, "MAE": metrics[h]["MAE"], "RMSE": metrics[h]["RMSE"]} for h in sorted(metrics)]
    pd.DataFrame(rows).to_csv(run_dir / f"{split_name}_metrics.csv", index=False)


def append_results_summary(model_name: str, run_dir: Path, test_metrics: dict):
    row = {
        "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model_name": model_name,
        "run_dir": str(run_dir),
        "tag": SHORT_TAG,
    }
    for h in EVAL_HORIZONS:
        row[f"test_MAE_{h}h"] = float(test_metrics[h]["MAE"])
        row[f"test_RMSE_{h}h"] = float(test_metrics[h]["RMSE"])

    df_new = pd.DataFrame([row])
    if RESULTS_SUMMARY_SHORT.exists():
        df_old = pd.read_csv(RESULTS_SUMMARY_SHORT)
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new

    df.to_csv(RESULTS_SUMMARY_SHORT, index=False)
    return RESULTS_SUMMARY_SHORT


@torch.inference_mode()
def collect_selected_horizon_predictions(model, loader, horizons=EVAL_HORIZONS):
    model.eval()
    pred_list = []
    true_list = []

    for xb, yb, tfb in tqdm(loader, desc="Collect preds", leave=False):
        xb  = xb.to(DEVICE, non_blocking=True)
        yb  = yb.to(DEVICE, non_blocking=True)
        tfb = tfb.to(DEVICE, non_blocking=True)

        pred = model(xb, tfb)
        pred_u = pred * flow_std_t + flow_mean_t
        true_u = yb   * flow_std_t + flow_mean_t

        pred_sel = []
        true_sel = []
        for h in horizons:
            idx = h - 1
            pred_sel.append(pred_u[:, idx, :].detach().cpu().numpy())
            true_sel.append(true_u[:, idx, :].detach().cpu().numpy())

        pred_sel = np.stack(pred_sel, axis=1)   # (B, H, N)
        true_sel = np.stack(true_sel, axis=1)   # (B, H, N)

        pred_list.append(pred_sel.astype(np.float32))
        true_list.append(true_sel.astype(np.float32))

    pred_all = np.concatenate(pred_list, axis=0)
    true_all = np.concatenate(true_list, axis=0)
    return pred_all, true_all


def train_deep_model(
    model_name: str,
    model: nn.Module,
    train_loader,
    val_loader,
    test_loader,
    epochs: int = 40,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    clip: float = 5.0,
    patience: int = 6,
    eval_every: int = 2,
    save_predictions: bool = True,
):
    run_dir = make_run_dir(model_name)
    print("Run dir:", run_dir)

    config = {
        "model_name": model_name,
        "seed": SEED,
        "in_len": IN_LEN,
        "out_len": OUT_LEN,
        "eval_horizons": EVAL_HORIZONS,
        "epochs": epochs,
        "lr": lr,
        "weight_decay": weight_decay,
        "clip": clip,
        "patience": patience,
        "eval_every": eval_every,
        "tag": SHORT_TAG,
    }
    save_json(run_dir / "config.json", config)

    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.SmoothL1Loss(beta=1.0)

    best_score = float("inf")
    best_state = None
    best_epoch = None
    bad_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0

        for xb, yb, tfb in tqdm(train_loader, desc=f"Train {epoch}/{epochs}", leave=False):
            xb  = xb.to(DEVICE, non_blocking=True)
            yb  = yb.to(DEVICE, non_blocking=True)
            tfb = tfb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb, tfb)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()

            running += float(loss.item())

        row = {"epoch": epoch, "train_loss": running / max(len(train_loader), 1)}

        if (epoch % eval_every == 0) or (epoch == epochs):
            val_metrics = eval_horizons_fast(model, val_loader)
            val_avg = avg_mae(val_metrics)

            row["val_avg_MAE"] = float(val_avg)
            for h in EVAL_HORIZONS:
                row[f"val_MAE_{h}h"] = float(val_metrics[h]["MAE"])
                row[f"val_RMSE_{h}h"] = float(val_metrics[h]["RMSE"])

            print(f"\nEpoch {epoch}: train_loss={row['train_loss']:.6f}  val_avg_MAE={val_avg:.3f}")
            print_metrics("Validation", val_metrics)

            if val_avg < best_score:
                best_score = float(val_avg)
                best_epoch = int(epoch)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                torch.save(best_state, run_dir / "best.pt")
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"\nEarly stopping. Best val_avg_MAE={best_score:.3f} at epoch {best_epoch}.")
                    history.append(row)
                    break

        history.append(row)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

    assert best_state is not None, "No best model was saved. Check eval_every and training loop."
    model.load_state_dict(best_state)

    print("\nEvaluating on TEST set...")
    test_metrics = eval_horizons_fast(model, test_loader)
    print_metrics(f"{model_name} — TEST", test_metrics)

    save_metrics_files(run_dir, "test", test_metrics)
    append_results_summary(model_name, run_dir, test_metrics)

    save_json(run_dir / "training_summary.json", {
        "best_val_avg_MAE": best_score,
        "best_epoch": best_epoch,
        "test_metrics": test_metrics,
    })

    if save_predictions:
        pred_u, true_u = collect_selected_horizon_predictions(model, test_loader, horizons=EVAL_HORIZONS)
        np.savez_compressed(
            run_dir / "test_pred_true_selected_horizons.npz",
            pred=pred_u.astype(np.float32),   # (samples, horizons, nodes)
            true=true_u.astype(np.float32),
            horizons=np.array(EVAL_HORIZONS, dtype=np.int32),
            stations=stations,
        )

    print("\nSaved outputs to:", run_dir)
    return run_dir, test_metrics

## Classical baselines: Elastic Net and Random Forest

These two models follow the same design used in the original notebook:

- fit one model per node
- use the past 24 hours of that node as historical predictors
- append future calendar features only for the report horizons
- predict the selected horizons directly as a multi-output regression problem

This keeps the shorter-horizon study aligned with the earlier methodology.

In [6]:
H_OFF = np.array([h - 1 for h in EVAL_HORIZONS], dtype=np.int64)
H = len(EVAL_HORIZONS)


def node_features_and_targets(node: int, starts: np.ndarray):
    # This preserves the same feature engineering pattern used in the original notebook.
    Xn = X_scaled[:, node, :]  # (T, F)

    # For a 2D input with axis=0, this returns shape (T-IN_LEN+1, F, IN_LEN).
    # We keep that ordering to stay methodologically identical.
    win = np.lib.stride_tricks.sliding_window_view(Xn, window_shape=IN_LEN, axis=0)
    X_hist = win[starts].reshape(len(starts), -1)  # (S, F * IN_LEN)

    idx = starts[:, None] + IN_LEN + H_OFF[None, :]
    X_tf = TF_all[idx].reshape(len(starts), -1)    # (S, H * 4)

    X_feat = np.concatenate([X_hist, X_tf], axis=1)
    y = Y_scaled[idx, node]                        # (S, H)

    return X_feat.astype(np.float32), y.astype(np.float32)


def save_classical_run_outputs(model_name: str, run_dir: Path, pred_u: np.ndarray, true_u: np.ndarray, metrics: dict):
    save_metrics_files(run_dir, "test", metrics)
    append_results_summary(model_name, run_dir, metrics)

    np.savez_compressed(
        run_dir / "test_pred_true_selected_horizons.npz",
        pred=pred_u.astype(np.float32),   # (samples, nodes, horizons)
        true=true_u.astype(np.float32),
        horizons=np.array(EVAL_HORIZONS, dtype=np.int32),
        stations=stations,
    )

### Elastic Net

In [6]:
def run_elasticnet_baseline(alpha=1e-3, l1_ratio=0.5, jobs=4):
    model_name = "ElasticNet"
    run_dir = make_run_dir(model_name)
    print("Run dir:", run_dir)

    save_json(run_dir / "config.json", {
        "model_name": model_name,
        "alpha": alpha,
        "l1_ratio": l1_ratio,
        "jobs": jobs,
        "eval_horizons": EVAL_HORIZONS,
        "tag": SHORT_TAG,
    })

    S_test = len(test_starts)
    pred_scaled = np.zeros((S_test, N, H), dtype=np.float32)
    true_scaled = np.zeros((S_test, N, H), dtype=np.float32)

    def fit_one(node: int):
        Xtr, ytr = node_features_and_targets(node, train_starts)
        Xte, yte = node_features_and_targets(node, test_starts)

        mdl = make_pipeline(
            StandardScaler(),
            MultiTaskElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=5000,
                random_state=SEED,
            ),
        )
        mdl.fit(Xtr, ytr)
        pred = mdl.predict(Xte).astype(np.float32)
        return node, pred, yte

    results = Parallel(n_jobs=jobs, prefer="threads")(
        delayed(fit_one)(node) for node in tqdm(range(N), desc="ElasticNet per-node")
    )

    for node, pred, yte in results:
        pred_scaled[:, node, :] = pred
        true_scaled[:, node, :] = yte

    pred_u = pred_scaled * flow_std[None, :, None] + flow_mean[None, :, None]
    true_u = true_scaled * flow_std[None, :, None] + flow_mean[None, :, None]

    metrics = {}
    for j, h in enumerate(EVAL_HORIZONS):
        err = pred_u[:, :, j] - true_u[:, :, j]
        metrics[h] = {
            "MAE": float(np.abs(err).mean()),
            "RMSE": float(np.sqrt((err ** 2).mean())),
        }

    print_metrics("ElasticNet — TEST", metrics)
    save_classical_run_outputs(model_name, run_dir, pred_u, true_u, metrics)
    return run_dir, metrics


# Run Elastic Net
run_dir_en, test_en = run_elasticnet_baseline(alpha=1e-3, l1_ratio=0.5, jobs=4)
print("Elastic Net run dir:", run_dir_en)

Run dir: artifacts/runs/20260320_003509_ElasticNet_short_h1_6_12_24


ElasticNet per-node:   0%|          | 0/1821 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:2412: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1.3010830879211426, tolerance: 0.42161405086517334
  ) = cd_fast.enet_coordinate_descent_multi_task(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:2412: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 0.4503946900367737, tolerance: 0.42117151618003845
  ) = cd_fast.enet_coordinate_descent_multi_task(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:2412: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1.4267100095748901, tolerance: 0.4216039478778839
  ) = cd_fast.enet_coordinate_descent_multi_task(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:24


ElasticNet — TEST
    1h  MAE=86.664  RMSE=162.562
    6h  MAE=146.991  RMSE=296.159
   12h  MAE=150.522  RMSE=303.586
   24h  MAE=148.584  RMSE=301.620
Elastic Net run dir: artifacts/runs/20260320_003509_ElasticNet_short_h1_6_12_24


### Random Forest

In [7]:
def run_random_forest_baseline(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=5,
    max_features="sqrt",
    jobs=4,
):
    model_name = "RandomForest"
    run_dir = make_run_dir(model_name)
    print("Run dir:", run_dir)

    save_json(run_dir / "config.json", {
        "model_name": model_name,
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_leaf": min_samples_leaf,
        "max_features": max_features,
        "jobs": jobs,
        "eval_horizons": EVAL_HORIZONS,
        "tag": SHORT_TAG,
    })

    S_test = len(test_starts)
    pred_scaled = np.zeros((S_test, N, H), dtype=np.float32)
    true_scaled = np.zeros((S_test, N, H), dtype=np.float32)

    def fit_one(node: int):
        Xtr, ytr = node_features_and_targets(node, train_starts)
        Xte, yte = node_features_and_targets(node, test_starts)

        mdl = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            n_jobs=1,   # parallelism is already across nodes
            random_state=SEED,
        )
        mdl.fit(Xtr, ytr)
        pred = mdl.predict(Xte).astype(np.float32)
        return node, pred, yte

    results = Parallel(n_jobs=jobs, prefer="threads")(
        delayed(fit_one)(node) for node in tqdm(range(N), desc="RF per-node")
    )

    for node, pred, yte in results:
        pred_scaled[:, node, :] = pred
        true_scaled[:, node, :] = yte

    pred_u = pred_scaled * flow_std[None, :, None] + flow_mean[None, :, None]
    true_u = true_scaled * flow_std[None, :, None] + flow_mean[None, :, None]

    metrics = {}
    for j, h in enumerate(EVAL_HORIZONS):
        err = pred_u[:, :, j] - true_u[:, :, j]
        metrics[h] = {
            "MAE": float(np.abs(err).mean()),
            "RMSE": float(np.sqrt((err ** 2).mean())),
        }

    print_metrics("RandomForest — TEST", metrics)
    save_classical_run_outputs(model_name, run_dir, pred_u, true_u, metrics)
    return run_dir, metrics


# Run Random Forest
run_dir_rf, test_rf = run_random_forest_baseline(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=5,
    max_features="sqrt",
    jobs=4,
)
print("Random Forest run dir:", run_dir_rf)

Run dir: artifacts/runs/20260320_015947_RandomForest_short_h1_6_12_24


RF per-node:   0%|          | 0/1821 [00:00<?, ?it/s]


RandomForest — TEST
    1h  MAE=101.893  RMSE=228.602
    6h  MAE=108.881  RMSE=247.180
   12h  MAE=113.874  RMSE=260.158
   24h  MAE=115.045  RMSE=258.949
Random Forest run dir: artifacts/runs/20260320_015947_RandomForest_short_h1_6_12_24


## LSTM 

In [8]:
class LSTM_Baseline(nn.Module):
    def __init__(
        self,
        in_dim: int,
        out_len: int,
        hidden: int = 64,
        layers: int = 1,
        dropout: float = 0.1,
        tf_dim: int = 4,
    ):
        super().__init__()
        self.out_len = out_len
        self.hidden = hidden

        self.lstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=hidden,
            num_layers=layers,
            dropout=(dropout if layers > 1 else 0.0),
            batch_first=True,
        )
        self.head = nn.Linear(hidden + tf_dim, 1)

    def forward(self, x, tf):
        # x: (B, F, N, IN) -> (B, N, IN, F) -> (B*N, IN, F)
        B, F, Nn, INL = x.shape
        x_seq = x.permute(0, 2, 3, 1).contiguous().view(B * Nn, INL, F)

        out, (h, c) = self.lstm(x_seq)
        h_last = h[-1]  # (B*N, hidden)

        tf_rep = (
            tf.unsqueeze(1)
              .expand(B, Nn, self.out_len, tf.shape[-1])
              .contiguous()
              .view(B * Nn, self.out_len, tf.shape[-1])
        )

        h_rep = h_last.unsqueeze(1).expand(B * Nn, self.out_len, self.hidden)
        z = torch.cat([h_rep, tf_rep], dim=-1)             # (B*N, OUT, hidden+4)
        y = self.head(z).squeeze(-1)                       # (B*N, OUT)
        y = y.view(B, Nn, self.out_len).permute(0, 2, 1)  # (B, OUT, N)
        return y


lstm_model = LSTM_Baseline(
    in_dim=Fdim,
    out_len=OUT_LEN,
    hidden=64,
    layers=1,
    dropout=0.1,
).to(DEVICE)

run_dir_lstm, test_lstm = train_deep_model(
    model_name="LSTM",
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    epochs=40,
    patience=6,
    eval_every=2,
)

print("LSTM run dir:", run_dir_lstm)
del lstm_model
clear_memory()

Run dir: artifacts/runs/20260320_020434_LSTM_short_h1_6_12_24


Train 1/40:   0%|          | 0/133 [00:00<?, ?it/s]

Train 2/40:   0%|          | 0/133 [00:00<?, ?it/s]

Eval:   0%|          | 0/43 [00:00<?, ?it/s]


Epoch 2: train_loss=0.359392  val_avg_MAE=406.214

Validation
    1h  MAE=400.755  RMSE=657.711
    6h  MAE=408.166  RMSE=669.604
   12h  MAE=409.118  RMSE=667.803
   24h  MAE=406.816  RMSE=667.879


...


Epoch 34: train_loss=0.149165  val_avg_MAE=204.023

Validation
    1h  MAE=198.720  RMSE=368.317
    6h  MAE=204.090  RMSE=376.491
   12h  MAE=205.919  RMSE=377.843
   24h  MAE=207.363  RMSE=380.917

Early stopping. Best val_avg_MAE=199.149 at epoch 22.

Evaluating on TEST set...


Eval:   0%|          | 0/91 [00:00<?, ?it/s]


LSTM — TEST
    1h  MAE=199.953  RMSE=373.778
    6h  MAE=206.222  RMSE=383.113
   12h  MAE=208.715  RMSE=385.985
   24h  MAE=206.868  RMSE=385.126


Collect preds:   0%|          | 0/91 [00:00<?, ?it/s]


Saved outputs to: artifacts/runs/20260320_020434_LSTM_short_h1_6_12_24
LSTM run dir: artifacts/runs/20260320_020434_LSTM_short_h1_6_12_24


## CNN–GRU–LSTM baseline

This is the hybrid sequence model from the paper 

- temporal Conv1D per node
- GRU encoder
- LSTM encoder
- direct multi-output horizon head
- optional future time bias shared across nodes

It already supports arbitrary `OUT_LEN`, so the short-horizon conversion is direct.

In [9]:
class CNN_GRU_LSTM_MultiHorizon(nn.Module):
    def __init__(
        self,
        in_dim: int,
        out_len: int,
        tf_dim: int = 4,
        conv_channels: int = 32,
        conv_kernel: int = 3,
        gru_hidden: int = 64,
        lstm_hidden: int = 64,
        dropout: float = 0.1,
        use_time_bias: bool = True,
        node_chunk: int = 256,
    ):
        super().__init__()
        self.in_dim = in_dim
        self.out_len = out_len
        self.tf_dim = tf_dim
        self.use_time_bias = use_time_bias
        self.node_chunk = node_chunk

        pad = conv_kernel // 2

        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=in_dim, out_channels=conv_channels, kernel_size=conv_kernel, padding=pad),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(in_channels=conv_channels, out_channels=conv_channels, kernel_size=conv_kernel, padding=pad),
            nn.ReLU(),
        )

        self.gru = nn.GRU(
            input_size=conv_channels,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
        )
        self.lstm = nn.LSTM(
            input_size=gru_hidden,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
        )

        self.h_to_out = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, out_len),
        )

        if use_time_bias:
            self.tf_to_bias = nn.Sequential(
                nn.Linear(tf_dim, 32),
                nn.ReLU(),
                nn.Linear(32, 1),
            )

    def forward(self, x, tf):
        B, Fdim_local, N_local, IN_local = x.shape
        device = x.device
        dtype = x.dtype

        if self.use_time_bias:
            tf = tf.to(device=device, dtype=dtype)
            time_bias = self.tf_to_bias(tf).squeeze(-1)  # (B, OUT)
        else:
            time_bias = None

        out = torch.empty((B, self.out_len, N_local), device=device, dtype=dtype)

        for s in range(0, N_local, self.node_chunk):
            e = min(N_local, s + self.node_chunk)
            Nc = e - s

            x_chunk = x[:, :, s:e, :]  # (B, F, Nc, IN)
            x_seq = x_chunk.permute(0, 2, 1, 3).contiguous().view(B * Nc, Fdim_local, IN_local)

            z = self.cnn(x_seq)               # (B*Nc, C, IN)
            z = z.transpose(1, 2).contiguous()

            z, _ = self.gru(z)
            z, (h, c) = self.lstm(z)
            h_last = h[-1]                   # (B*Nc, lstm_hidden)

            pred = self.h_to_out(h_last)     # (B*Nc, OUT)
            pred = pred.view(B, Nc, self.out_len).permute(0, 2, 1).contiguous()

            if time_bias is not None:
                pred = pred + time_bias.unsqueeze(-1)

            out[:, :, s:e] = pred

        return out


cnn_gru_lstm_model = CNN_GRU_LSTM_MultiHorizon(
    in_dim=Fdim,
    out_len=OUT_LEN,
    tf_dim=4,
    conv_channels=32,
    conv_kernel=3,
    gru_hidden=64,
    lstm_hidden=64,
    dropout=0.1,
    use_time_bias=True,
    node_chunk=256,
).to(DEVICE)

run_dir_cnn_gru_lstm, test_cnn_gru_lstm = train_deep_model(
    model_name="CNN_GRU_LSTM",
    model=cnn_gru_lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    epochs=40,
    patience=6,
    eval_every=2,
)

print("CNN-GRU-LSTM run dir:", run_dir_cnn_gru_lstm)
del cnn_gru_lstm_model
clear_memory()

Run dir: artifacts/runs/20260320_021318_CNN_GRU_LSTM_short_h1_6_12_24


Train 1/40:   0%|          | 0/133 [00:00<?, ?it/s]

Train 2/40:   0%|          | 0/133 [00:00<?, ?it/s]

Eval:   0%|          | 0/43 [00:00<?, ?it/s]


Epoch 2: train_loss=0.107290  val_avg_MAE=174.629

Validation
    1h  MAE=167.251  RMSE=302.017
    6h  MAE=177.823  RMSE=321.225
   12h  MAE=171.884  RMSE=322.849
   24h  MAE=181.557  RMSE=335.163


...


Epoch 40: train_loss=0.059176  val_avg_MAE=113.020

Validation
    1h  MAE=87.674  RMSE=166.638
    6h  MAE=120.761  RMSE=241.440
   12h  MAE=122.033  RMSE=250.146
   24h  MAE=121.611  RMSE=253.652

Evaluating on TEST set...


Eval:   0%|          | 0/91 [00:00<?, ?it/s]


CNN_GRU_LSTM — TEST
    1h  MAE=87.030  RMSE=166.164
    6h  MAE=111.816  RMSE=223.259
   12h  MAE=115.437  RMSE=234.806
   24h  MAE=119.707  RMSE=245.041


Collect preds:   0%|          | 0/91 [00:00<?, ?it/s]


Saved outputs to: artifacts/runs/20260320_021318_CNN_GRU_LSTM_short_h1_6_12_24
CNN-GRU-LSTM run dir: artifacts/runs/20260320_021318_CNN_GRU_LSTM_short_h1_6_12_24


## GraphWaveNet family

In [10]:
def require_col(df: pd.DataFrame, candidates, friendly_name: str):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"Could not find column for '{friendly_name}'. Tried: {candidates}. "
        f"Available columns: {list(df.columns)}"
    )


assert TRAFFIC_CSV.exists(), f"Missing {TRAFFIC_CSV}"
assert META_XLSX.exists(), f"Missing {META_XLSX}"

traffic_raw = pd.read_csv(TRAFFIC_CSV)
meta_raw = pd.read_excel(META_XLSX)

traffic_raw.columns = [str(c).strip() for c in traffic_raw.columns]
meta_raw.columns = [str(c).strip() for c in meta_raw.columns]

st_col_traffic = require_col(traffic_raw, ["Station", "station", "ID"], "traffic station id")
dir_col_traffic = require_col(traffic_raw, ["Direction of Travel", "direction", "Dir"], "traffic direction")

tmp = traffic_raw[[st_col_traffic, dir_col_traffic]].copy()
tmp = tmp.rename(columns={st_col_traffic: "station", dir_col_traffic: "direction"})
tmp["station"] = pd.to_numeric(tmp["station"], errors="coerce").astype("Int64")
tmp = tmp.dropna(subset=["station"])
tmp["station"] = tmp["station"].astype(int)

station_dir = tmp.groupby("station")["direction"].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]
)
station_dir = station_dir.reindex(stations)

meta_id_col = require_col(meta_raw, ["station", "ID", "Station"], "metadata station id")
meta_fwy_col = require_col(meta_raw, ["Fwy", "FWY", "fwy", "Freeway"], "metadata freeway")
meta_pm_col = require_col(meta_raw, ["Abs PM", "abs_pm", "AbsPM", "Postmile", "PM"], "metadata absolute postmile")

meta = meta_raw.rename(columns={
    meta_id_col: "station",
    meta_fwy_col: "Fwy",
    meta_pm_col: "Abs PM",
}).copy()

meta["station"] = pd.to_numeric(meta["station"], errors="coerce").astype("Int64")
meta = meta.dropna(subset=["station"]).copy()
meta["station"] = meta["station"].astype(int)


def build_adjacency_fwy_dir(meta_df, stations, station_dir, k_neighbors=4):
    meta_sub = meta_df[meta_df["station"].isin(stations)].copy()
    meta_sub["fwy"] = meta_sub["Fwy"].astype(str)
    meta_sub["abs_pm"] = pd.to_numeric(meta_sub["Abs PM"], errors="coerce")
    meta_sub["direction"] = meta_sub["station"].map(station_dir)

    meta_sub = meta_sub.dropna(subset=["abs_pm", "direction"]).copy()
    meta_sub["direction"] = meta_sub["direction"].astype(str)

    station_to_idx = {s: i for i, s in enumerate(stations)}
    N_local = len(stations)
    A_dir = np.zeros((N_local, N_local), dtype=np.float32)

    all_dists = []
    for (_, _), grp in meta_sub.sort_values(["fwy", "direction", "abs_pm"]).groupby(["fwy", "direction"]):
        pm = grp["abs_pm"].values
        if len(pm) < 2:
            continue
        d = np.diff(np.sort(pm))
        d = d[d > 0]
        all_dists.extend(d.tolist())

    sigma = float(np.median(all_dists)) if len(all_dists) else 0.5
    sigma = max(sigma, 1e-3)

    def weight(dist):
        return float(np.exp(-(dist ** 2) / (sigma ** 2)))

    for (_, _), grp in meta_sub.sort_values(["fwy", "direction", "abs_pm"]).groupby(["fwy", "direction"]):
        grp = grp.sort_values("abs_pm")
        ids = grp["station"].astype(int).tolist()
        pms = grp["abs_pm"].astype(float).tolist()

        for i, sid in enumerate(ids):
            ii = station_to_idx[sid]
            for step in range(1, k_neighbors + 1):
                if i - step >= 0:
                    sj = ids[i - step]
                    jj = station_to_idx[sj]
                    A_dir[ii, jj] = weight(abs(pms[i] - pms[i - step]))
                if i + step < len(ids):
                    sj = ids[i + step]
                    jj = station_to_idx[sj]
                    A_dir[ii, jj] = weight(abs(pms[i] - pms[i + step]))

    np.fill_diagonal(A_dir, 1.0)
    A_dir = np.maximum(A_dir, A_dir.T)
    return A_dir


def row_normalize_dense(A_dense, eps=1e-6):
    d = A_dense.sum(axis=1, keepdims=True)
    return A_dense / (d + eps)


def dense_to_sparse(A_dense, device):
    idx = np.nonzero(A_dense)
    values = A_dense[idx].astype(np.float32)
    indices = np.vstack(idx)

    return torch.sparse_coo_tensor(
        torch.tensor(indices, dtype=torch.long, device=device),
        torch.tensor(values, dtype=torch.float32, device=device),
        size=A_dense.shape,
        device=device,
    ).coalesce()


A_dir = build_adjacency_fwy_dir(meta, stations, station_dir, k_neighbors=4)
A_rw  = row_normalize_dense(A_dir)
A_rwT = row_normalize_dense(A_dir.T)

supports = [
    dense_to_sparse(A_rw, DEVICE),
    dense_to_sparse(A_rwT, DEVICE),
]

print("Directed adjacency shape:", A_dir.shape)
print("Directed adjacency density:", float((A_dir > 0).mean()))
print("Support nnz:", [int(s._nnz()) for s in supports])

Directed adjacency shape: (1821, 1821)
Directed adjacency density: 0.0038374073179432942
Support nnz: [12720, 12720]


In [11]:
class NConv(nn.Module):
    def forward(self, x, A_sp):
        # x: (B, C, N, T)
        B, C, N_local, T_local = x.shape
        x_r = x.permute(2, 0, 1, 3).reshape(N_local, -1).float()
        out = torch.sparse.mm(A_sp, x_r)
        out = out.reshape(N_local, B, C, T_local).permute(1, 2, 0, 3)
        return out.to(dtype=x.dtype)


class DiffusionGraphConv(nn.Module):
    def __init__(self, c_in, c_out, supports, order=1, dropout=0.0):
        super().__init__()
        self.nconv = NConv()
        self.supports = supports
        self.order = order
        self.dropout = dropout

        c_total = c_in * (1 + len(supports) * order)
        self.mlp = nn.Conv2d(c_total, c_out, kernel_size=(1, 1))

    def forward(self, x):
        out = [x]
        for A_sp in self.supports:
            x1 = self.nconv(x, A_sp)
            out.append(x1)
            for _ in range(2, self.order + 1):
                x1 = self.nconv(x1, A_sp)
                out.append(x1)

        h = torch.cat(out, dim=1)
        h = self.mlp(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return h


class CausalConv2d(nn.Module):
    def __init__(self, c_in, c_out, kernel_size=2, dilation=1):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv2d(
            c_in,
            c_out,
            kernel_size=(1, kernel_size),
            dilation=(1, dilation),
        )

    def forward(self, x):
        x = F.pad(x, (self.pad, 0, 0, 0))
        return self.conv(x)


class GraphWaveNetEncoder(nn.Module):
    def __init__(
        self,
        num_nodes,
        in_dim,
        supports,
        residual_channels=32,
        dilation_channels=32,
        skip_channels=64,
        end_channels=128,
        kernel_size=2,
        blocks=2,
        layers_per_block=4,
        gcn_order=1,
        dropout=0.1,
    ):
        super().__init__()
        self.dropout = dropout
        self.kernel_size = kernel_size
        self.blocks = blocks
        self.layers_per_block = layers_per_block

        receptive_field = 1
        for _ in range(blocks):
            for i in range(layers_per_block):
                receptive_field += (kernel_size - 1) * (2 ** i)
        self.receptive_field = receptive_field

        self.start_conv = nn.Conv2d(in_dim, residual_channels, kernel_size=(1, 1))

        self.filter_convs = nn.ModuleList()
        self.gate_convs   = nn.ModuleList()
        self.skip_convs   = nn.ModuleList()
        self.bn           = nn.ModuleList()
        self.gconvs       = nn.ModuleList()

        for _ in range(blocks):
            for i in range(layers_per_block):
                dilation = 2 ** i
                self.filter_convs.append(CausalConv2d(residual_channels, dilation_channels, kernel_size, dilation))
                self.gate_convs.append(CausalConv2d(residual_channels, dilation_channels, kernel_size, dilation))
                self.skip_convs.append(nn.Conv2d(dilation_channels, skip_channels, kernel_size=(1, 1)))
                self.gconvs.append(
                    DiffusionGraphConv(
                        dilation_channels,
                        residual_channels,
                        supports,
                        order=gcn_order,
                        dropout=dropout,
                    )
                )
                self.bn.append(nn.BatchNorm2d(residual_channels))

        self.end_conv_1 = nn.Conv2d(skip_channels, end_channels, kernel_size=(1, 1))

    def forward(self, x):
        if x.size(-1) < self.receptive_field:
            pad_len = self.receptive_field - x.size(-1)
            x = F.pad(x, (pad_len, 0, 0, 0))

        x = self.start_conv(x)
        skip = None

        for i in range(len(self.filter_convs)):
            residual = x

            filt = torch.tanh(self.filter_convs[i](x))
            gate = torch.sigmoid(self.gate_convs[i](x))
            x = filt * gate
            x = F.dropout(x, p=self.dropout, training=self.training)

            s = self.skip_convs[i](x)
            skip = s if skip is None else (skip + s)

            x = self.gconvs[i](x)
            x = x + residual
            x = self.bn[i](x)

        x = F.relu(skip)
        x = F.relu(self.end_conv_1(x))
        return x  # (B, end_channels, N, T)


class GraphWaveNetRNN(nn.Module):
    def __init__(
        self,
        num_nodes,
        in_dim,
        out_len,
        supports,
        end_channels=128,
        use_gru=False,
        use_lstm=False,
        rnn_hidden=128,
        dropout=0.1,
        **encoder_kwargs,
    ):
        super().__init__()
        self.out_len = out_len
        self.use_gru = use_gru
        self.use_lstm = use_lstm

        self.encoder = GraphWaveNetEncoder(
            num_nodes=num_nodes,
            in_dim=in_dim,
            supports=supports,
            end_channels=end_channels,
            dropout=dropout,
            **encoder_kwargs,
        )

        if use_gru:
            self.gru = nn.GRU(input_size=end_channels, hidden_size=rnn_hidden, batch_first=True)
        else:
            self.gru = None

        if use_lstm:
            self.lstm = nn.LSTM(
                input_size=(rnn_hidden if use_gru else end_channels),
                hidden_size=rnn_hidden,
                batch_first=True,
            )
        else:
            self.lstm = None

        final_dim = rnn_hidden if (use_gru or use_lstm) else end_channels

        self.time_embed = nn.Linear(4, final_dim)
        self.horizon_out = nn.Linear(final_dim, 1)

    def forward(self, x, tf_future):
        h = self.encoder(x)                            # (B, C, N, T)
        B, C, N_local, T_local = h.shape

        h_seq = h.permute(0, 2, 3, 1).contiguous()    # (B, N, T, C)
        h_seq = h_seq.view(B * N_local, T_local, C)   # (B*N, T, C)

        if self.gru is not None:
            h_seq, _ = self.gru(h_seq)

        if self.lstm is not None:
            h_seq, _ = self.lstm(h_seq)

        last = h_seq[:, -1, :]                        # (B*N, D)
        z = last.view(B, N_local, -1)                 # (B, N, D)

        te = self.time_embed(tf_future)               # (B, OUT, D)
        out = F.relu(z.unsqueeze(1) + te.unsqueeze(2))
        out = self.horizon_out(out).squeeze(-1)       # (B, OUT, N)
        return out


COMMON_GWN_KWARGS = dict(
    num_nodes=N,
    in_dim=Fdim,
    out_len=OUT_LEN,
    supports=supports,
    end_channels=128,
    rnn_hidden=128,
    dropout=0.1,
    residual_channels=32,
    dilation_channels=32,
    skip_channels=64,
    kernel_size=2,
    blocks=2,
    layers_per_block=4,
    gcn_order=1,
)

# Sanity batch for shape checks
xb, yb, tfb = next(iter(train_loader))

gwn_base_model = GraphWaveNetRNN(
    **COMMON_GWN_KWARGS,
    use_gru=False,
    use_lstm=False,
).to(DEVICE)

with torch.no_grad():
    out = gwn_base_model(xb.to(DEVICE), tfb.to(DEVICE))
print("GraphWaveNet sanity output:", out.shape)

clear_memory()

GraphWaveNet sanity output: torch.Size([8, 24, 1821])
